### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/01_Freeze_Experiment.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Fnotebooks%2Fthesis%2F01_Freeze_Experiment.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Fnotebooks%2Fthesis%2F01_Freeze_Experiment.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/01_Freeze_Experiment.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.

Frozen training and collection notebooks also require the campaign artifacts prepared in notebook **01**.

See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.


In [ ]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME,
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "notebooks" / "init.py" for path in _locations
                     if (path / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, repository=REPOSITORY, revision=REVISION,
    runtime=RUNTIME, cuda=CUDA,
)


# 01 — Freeze the TensorFlow 2.20 campaign

Run after notebook00 shows adequate learning and acceptable cost. Select a **TensorFlow 2.20 / Keras 3** kernel. This notebook prepares configurations and a checklist; it trains no model.

In [ ]:
from IPython.display import display

from notebooks.thesis.workflow import check_runtime


print(check_runtime())
from notebooks.thesis.workflow import prepare_campaign, campaign_checklist


CAMPAIGN = ROOT / "results/thesis_route_one/minimum_v5_tf220"

### 1. Review the design

CIFAR-10: 5 two-class tasks, platform/extra joint/learned. CIFAR-100: 10 ten-class tasks, those methods plus random/CE only. Three paired seeds give **9 + 15 = 24 complete streams**. Seed17 remains development-only. The primary comparison is learned minus extra joint; report every stream, mean, sample SD and the native paired interval.

This tests a **TMCL-inspired classifier adaptation**. It does not reproduce TMCL scores or test noise-dependent consolidation. Earlier HPO used official test data. This campaign is a **test-informed benchmark**, not independent confirmation; its selection history is retained in benchmark_selection.json.

In [ ]:
SEEDS = [1103, 2207, 3301]
TEMPLATES = {dataset: ROOT / "notebooks/thesis/configs" / f"{dataset}.yaml"
             for dataset in ("cifar10", "cifar100")}
print("Campaign:", CAMPAIGN)
print("Paired seeds:", SEEDS, "— 24 complete streams")

### 2. Freeze once

Finish source/recipe edits first. Existing campaigns are preserved; preparation refuses an existing directory. Keep an independent unchanged copy of frozen_design.json.

The approved recipe uses joint Adam LR 0.001, semantic Adam LR 0.0003, and primary-head-only timestep ensemble. Prior official-test HPO is explicitly disclosed. Benchmark mode freezes the same run/source/metric identities as confirmation, while making no claim of test-independent selection. New seeds do not restore that independence.

In [ ]:
import json
SELECTION_PROVENANCE = json.loads(
    (ROOT / "notebooks/thesis/benchmark_selection.json").read_text(encoding="utf-8"))
record_path = prepare_campaign(CAMPAIGN, TEMPLATES, SEEDS,
                               phase="benchmark", selection_provenance=SELECTION_PROVENANCE)
print("Frozen benchmark record:", record_path)

### 3. Follow this checklist

Run its rows in order, one named notebook in a fresh kernel at a time. Conditions are randomized within each paired stream. A failed or unfavorable run must not be silently skipped.

In [ ]:
checklist = campaign_checklist(record_path)
display(checklist)
print("Saved:", CAMPAIGN / "execution_checklist.csv")